In [23]:
from ast import *
from utils import *
import import_ipynb
from x86_ast import *
from rco_test import *
from select_instr import *

In [24]:
 def assign_homes_arg(a: arg, home: Dict[Variable, arg]) -> arg:
        # YOUR CODE HERE
        match a:
            case Variable(val):               
                return Deref('rbp',home[Variable(val)] )
            case Immediate(val):
                return Immediate(val)
            case Reg(reg):
                return Reg(reg)
            case _:
                return arg
        

In [ ]:
 def assign_homes_instr(i: instr,
                           home: Dict[Variable, arg]) -> instr:
        # YOUR CODE HERE
        
        match i:
            case Instr('movq',[Immediate(int),Variable(id)]):
                if Variable(id) not in home:
                    home[Variable(id)] =  ((len(home) + 1)  * -8)
                return Instr('movq',[assign_homes_arg(Immediate(int),home) , assign_homes_arg(Variable(id), home)])
            case Instr('movq', [Variable(id1), Variable(id2)]):
                if Variable(id1) not in home:
                    home[Variable(id1)] = ((len(home) + 1) * -8)
                if Variable(id2) not in home:
                    home[Variable(id2)] = ((len(home) + 1) * -8)
                return Instr('movq',[assign_homes_arg(Variable(id1), home), assign_homes_arg(Variable(id2),home)])
            case Instr('movq', [Variable(id), Reg(reg)]):
                if Variable(id) not in home:
                     home[Variable(id)] = ((len(home) + 1) * -8)
                return Instr('movq', [assign_homes_arg(Variable(id),home), assign_homes_arg(Reg(reg),home)] )
            case Instr('movq',[Reg(reg), Variable(id)]):
                  if Variable(id) not in home:
                       home[Variable(id)] = ((len(home) + 1) * -8)
                  return Instr('movq',[assign_homes_arg(Reg(reg),home), assign_homes_arg(Variable(id),home)])
            case Instr('addq',[Immediate(int), Variable(id)]):
                if Variable(id) not in home:
                    home[Variable(id)] = ((len(home) + 1) * -8)
                return Instr('addq',[assign_homes_arg(Immediate(int),home), assign_homes_arg(Variable(id),home)])
            case Instr('subq',[Immediate(int), Variable(id)]):
                if Variable(id) not in home:
                    home[Variable(id)] = ((len(home) + 1) * -8)
                return Instr('subq',[assign_homes_arg(Immediate(int),home), assign_homes_arg(Variable(id),home)])
            case Instr('addq',[Variable(id), Reg(reg)]):
                if Variable(id) not in home:
                    home[Variable(id)] = ((len(home)+1) * -8)
                return Instr('addq',[assign_homes_arg(Variable(id),home), assign_homes_arg(Reg(reg),home)])
            case Instr('subq',[Variable(id),Reg(reg)]):
                if Variable(id) not in home:
                    home[Variable(id)] = ((len(home) + 1) * -8)
                return Instr('subq',[assign_homes_arg(Variable(id),home), assign_homes_arg(Reg(reg),home)])
            case Instr('addq',[Variable(id1),Variable(id2)]):
                if Variable(id1) not in home:
                    home[Variable(id1)] = ((len(home) + 1) * -8)
                if Variable(id2) not in home:
                    home[Variable(id2)] = ((len(home)+1) * -8)
                return Instr('addq',[assign_homes_arg(Variable(id1),home), assign_homes_arg(Variable(id2),home)])
            case Instr('subq',[Variable(id1),Variable(id2)]):
                if Variable(id1) not in home:
                    home[Variable(id1)] = ((len(home) + 1) * -8)
                if Variable(id2) not in home:
                    home[Variable(id2)] = ((len(home) + 1) * -8)
                return Instr('subq',[assign_homes_arg(Variable(id1),home) , assign_homes_arg(Variable(id2),home)])
            
            case _:
                return i
                
                
                


In [26]:
 def assign_homes(p: X86Program) -> X86Program:
        # YOUR CODE HERE
    
        new_list = []
        home = {}
        for instr in p.body:          
                    new_list.append(assign_homes_instr(instr,home))
        return X86Program(new_list)

In [28]:
if __name__ == "__main__":
    import textwrap
    code = textwrap.dedent("""
    a = 42 + -90
    b = a
    print(b)""")
    parsed_code = parse(code)
    rco_code = remove_complex_operands(parsed_code)
    select_instr_code = select_instruction(rco_code)
    print(select_instr_code)
    assign_homes_code = assign_homes(select_instr_code)
    print(assign_homes_code)

	.globl main
main:
    movq $90, %rax
    negq %rax
    movq %rax, temp.0
    movq $42, %rax
    addq temp.0, %rax
    movq %rax, a
    movq a, b
    movq b, %rdi
    callq print_int


	.globl main
main:
    movq $90, %rax
    negq %rax
    movq %rax, -8(%rbp)
    movq $42, %rax
    addq -8(%rbp), %rax
    movq %rax, -16(%rbp)
    movq -16(%rbp), -24(%rbp)
    movq -24(%rbp), %rdi
    callq print_int


